# Kaggle - Buoc 2: Inference + Submit

Chay hybrid pipeline (rules + LLM + LoRA) tren toan bo test set, tao file zip submit.

**TRUOC KHI CHAY:**
1. Upload test set len Kaggle input
2. Chay xong `kaggle_sft_train.ipynb` de co `lora_adapter/`

**Thu tu thuc hien:**
1. Cell 1-2: Cai dat + clone repo
2. Cell 3: Cau hinh
3. Cell 4: Load model + adapter
4. Cell 5: Chay hybrid inference
5. Cell 6: Tao file submit

## Cell 1 - Cai dat thu vien

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes
!pip install -q editdistance

## Cell 2 - Clone repo tu GitHub

In [ ]:
# ===== THAY DOI Ở ĐÂY =====
GITHUB_USER = "bphong-cyrus"
REPO_NAME = "Viettel-AI-Race"
# ===========================

import os, sys
WORK_DIR = "/kaggle/working"
REPO_DIR = f"{WORK_DIR}/{REPO_NAME}"

if os.path.exists(REPO_DIR):
    print(f"Repo da ton tai: {REPO_DIR}")
else:
    print(f"Cloning https://github.com/{GITHUB_USER}/{REPO_NAME}.git")
    os.system(f"cd {WORK_DIR} && git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git")

os.chdir(REPO_DIR)
if '/kaggle/working/Viettel-AI-Race' not in sys.path:
    sys.path.insert(0, '/kaggle/working/Viettel-AI-Race')

print(f"Working: {os.getcwd()}")
print(f"Files: {sorted(os.listdir('.'))[:15]}")

## Cell 3 - Cau hinh inference

In [ ]:
# ===== CAU HINH Inference =====

# Base model - PHAI TRUNG voi model da train o buoc 1
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

# LoRA adapter tu kaggle_sft_train.ipynb
LORA_ADAPTER = "/kaggle/working/lora_adapter"

# Duong dan test set INPUT
# Upload test set len Kaggle, roi chinh duong dan o day
TEST_INPUT_DIR = "input/input"

# Pipeline mode
USE_HYBRID = True   # True = rules + LLM, False = LLM only
RULE_WEIGHT = 0.6    # Do tin cua rules trong hybrid

# Output
OUTPUT_DIR = "/kaggle/working/output_hybrid"
ZIP_PATH = "/kaggle/working/output_hybrid.zip"

print(f"Base model : {BASE_MODEL}")
print(f"LoRA       : {LORA_ADAPTER}")
print(f"Test input : {TEST_INPUT_DIR}")
print(f"Mode       : {'HYBRID' if USE_HYBRID else 'LLM-only'} (rule_weight={RULE_WEIGHT})")
print(f"Output     : {OUTPUT_DIR}")

## Cell 4 - Load model + adapter

In [ ]:
import torch, os, sys
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

sys.path.insert(0, '/kaggle/working/Viettel-AI-Race')

# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model (4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

if LORA_ADAPTER and os.path.exists(LORA_ADAPTER):
    print(f"Loading LoRA adapter: {LORA_ADAPTER}")
    model = PeftModel.from_pretrained(model, LORA_ADAPTER)
    print("LoRA adapter loaded")
else:
    print("WARNING: No LoRA adapter - using base model (zero-shot)")

model.eval()
print("Model ready for inference")

## Cell 5 - Chay Hybrid Inference

In [ ]:
import json, time, re
from pathlib import Path
import v20_pipeline
from hybrid_pipeline import merge_rule_and_llm

sys.path.insert(0, '/kaggle/working/Viettel-AI-Race')
from llm_prompts import SYSTEM_PROMPT

MAX_SEQ_LEN = 4096
MAX_NEW_TOKENS = 1500


def generate_with_model(text):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Trich xuat thuc the y te tu:\n{text}\n\nJSON:"},
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.05,
        )
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)


def parse_llm_output(raw_output, original_text):
    patterns = [
        r'\[\s*\{[\s\S]*?\}\s*\]',
        r'\{[\s\S]*"entities"[\s\S]*\}',
    ]
    for pat in patterns:
        m = re.search(pat, raw_output)
        if m:
            try:
                parsed = json.loads(m.group())
                if isinstance(parsed, dict) and "entities" in parsed:
                    parsed = parsed["entities"]
                if isinstance(parsed, list):
                    return parsed
            except:
                continue
    return []


def validate_and_fix(entity, text):
    e = dict(entity)
    if "text" not in e or "type" not in e:
        return None
    etext = e["text"]
    if "start" in e and "end" in e:
        if text[e["start"]:e["end"]] == etext:
            return e
        idx = text.find(etext)
        if idx >= 0:
            e["start"] = idx
            e["end"] = idx + len(etext)
            return e
    else:
        idx = text.find(etext)
        if idx >= 0:
            e["start"] = idx
            e["end"] = idx + len(etext)
        return e


def clean_entities(entities, text):
    VALID_TYPES = {"THUOC", "TRIEU_CHUNG", "CHAN_DOAN"}
    STOPWORDS = {"khong", "khong co", "neu", "nao", "nay", "benh nhan", "nam", "nu", "tuoi"}
    result = []
    seen = set()
    for e in entities:
        if e.get("type") not in VALID_TYPES:
            continue
        etext = e.get("text", "").strip()
        if len(etext) < 2:
            continue
        if etext.lower() in STOPWORDS:
            continue
        e2 = validate_and_fix(e, text)
        if e2 is None:
            continue
        key = (e2["type"], e2["start"], e2["end"])
        if key not in seen:
            seen.add(key)
            result.append(e2)
    return result


def extract_llm(text):
    raw = generate_with_model(text[:1800])
    entities = parse_llm_output(raw, text)
    return clean_entities(entities, text)


def hybrid_extract(text):
    rule_ents = v20_pipeline.extract_entities_v20(text)
    llm_ents = extract_llm(text)
    return merge_rule_and_llm(rule_ents, llm_ents, rule_weight=RULE_WEIGHT)


extractor = hybrid_extract if USE_HYBRID else extract_llm
mode_label = "HYBRID" if USE_HYBRID else "LLM-only"
print(f"Running: {mode_label} (rule_weight={RULE_WEIGHT})")

os.makedirs(OUTPUT_DIR, exist_ok=True)
test_files = sorted(Path(TEST_INPUT_DIR).glob("*.txt"),
                     key=lambda p: int(p.stem) if p.stem.isdigit() else 999)

print(f"\nProcessing {len(test_files)} files...")
t0 = time.time()
total_ents = 0
type_counts = {"THUOC": 0, "TRIEU_CHUNG": 0, "CHAN_DOAN": 0}

for i, fpath in enumerate(test_files, 1):
    text = fpath.read_text(encoding="utf-8")
    ents = extractor(text)
    with open(f"{OUTPUT_DIR}/{fpath.stem}.json", "w", encoding="utf-8") as f:
        json.dump(ents, f, ensure_ascii=False, indent=2)
    total_ents += len(ents)
    for e in ents:
        type_counts[e.get("type", "?")] = type_counts.get(e.get("type", "?"), 0) + 1
    if i % 20 == 0 or i == len(test_files):
        elapsed = time.time() - t0
        eta = elapsed / i * (len(test_files) - i)
        print(f"  [{i}/{len(test_files)}] {elapsed:.0f}s elapsed, ETA {eta:.0f}s | ents={total_ents}")

print(f"\nHoan tat! {len(test_files)} files")
print(f"Total entities: {total_ents}")
print(f"By type: {type_counts}")
print(f"Time: {time.time()-t0:.0f}s")
print(f"Output: {OUTPUT_DIR}")

## Cell 6 - Tao file submit .zip

In [ ]:
import zipfile, os
from pathlib import Path

print(f"Creating {ZIP_PATH} from {OUTPUT_DIR}...")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(Path(OUTPUT_DIR).glob("*.json")):
        zf.write(f, f.name)

zip_size = os.path.getsize(ZIP_PATH) / 1024
n_files = len(list(Path(OUTPUT_DIR).glob("*.json")))

print(f"ZIP created: {ZIP_PATH}")
print(f"Files: {n_files}, Size: {zip_size:.1f} KB")

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    names = zf.namelist()
    print(f"Verified: {len(names)} entries in zip")
    print(f"Sample: {sorted(names)[:5]}")

print("\nSAN SANG SUBMIT!")
print(f"Download: {ZIP_PATH}")